In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import trackio
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

keys = 'u', 'v', 'w', 'p_total'

def extract(real, keys):
    with h5py.File("/data1/dataToYouri/couette_dns_ml_f32.h5", 'r') as f:
        data = []
        for k in keys:
            data.append(np.array(f[real][k]))
        return np.stack(data, axis=1)

realizations = ['R13', 'R14', 'R15', 'R18', 'R19', 'R21', 'R22', 'R23', 'R26', 'R28', 'R3', 'R4', 'R5']
samplings = {real: np.s_[:-1:5000 // 300] if real in ["R3", "R4", "R5"] else np.s_[:] for real in realizations}

raw_d = {real: extract(real, keys)[samplings[real]] for real in realizations}
all_data = np.concatenate(list(raw_d.values()), axis=0)
T, C, X, Y, Z = all_data.shape
print("all_data", all_data.shape)

# Test and val realizations choosen as they have roughly avg mean
val_realizations = ["R18", "R5"]
test_realizations = ["R3", "R14"]  # ONLY USE FOR FINAL BENCHMARK, ANY DECISION TAKEN AFTER PERFORMANCE IS ESTABLISHED ON THIS DS IS LEAKAGE
train_realizations = [real for real in realizations if real not in val_realizations + test_realizations]
val_npds = np.concatenate([raw_d[r] for r in val_realizations])
train_npds = np.concatenate([raw_d[r] for r in train_realizations])

device = "cuda"

class DS(Dataset):
    def __init__(self, d):
        self.x = torch.from_numpy(d[:, 3:, :, 0, :]).float().to(device)
        self.y = torch.from_numpy(d[:, :3, :, :, :]).float().to(device)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]

# Per-(component, y) mean/std: the "fluctuation" space the network operates in.
normalization_axes = (0, 2, 4,)
all_data_mean = all_data.mean(normalization_axes, keepdims=True)
all_data_std = all_data.std(normalization_axes, keepdims=True) + 1e-10

all_data_mean_gpu = torch.from_numpy(all_data_mean).to(device)
all_data_std_gpu = torch.from_numpy(all_data_std).to(device)

def normalize(data):
    return (data - all_data_mean) / all_data_std

def unnormalize_velocity_gpu(data):
    return (data * all_data_std_gpu[:, :3]) + all_data_mean_gpu[:, :3]

val_ds = DS(normalize(val_npds))
train_ds = DS(normalize(train_npds))

def predict(model, flat_x_ds):
    model.eval()
    with torch.no_grad():
        return model(flat_x_ds)

batch_size = 4
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)
print("train", len(train_dl), "val", len(val_dl))


all_data (3900, 4, 24, 33, 24)
train 675 val 150


In [2]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding="same"),
            nn.SiLU(),
            nn.Conv3d(out_ch, out_ch, 3, padding="same"),
            nn.SiLU(),
        )
    def forward(self, x):
        return self.conv(x)


class M4(nn.Module):
    def __init__(self):
        super().__init__()
        self.pressure_encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding="same"),
            nn.SiLU(),
            nn.Conv2d(8, 32, 3, padding="same"),
            nn.SiLU(),
        )

        self.D = 33
        self.c3d = 32
        self.lift = nn.Conv2d(32, self.D * self.c3d, 3, padding="same")

        self.xd = 24
        self.zd = 24

        self.conv_blocks = nn.Sequential(
            *[ConvBlock(self.c3d, self.c3d) for _ in range(5)]
        )
        self.output = nn.Conv3d(self.c3d, 3, 3, padding="same")

    def forward(self, pressure):
        x = self.pressure_encoder(pressure)
        x = self.lift(x)
        # reshape into Batch, channels, x, D, z
        x = x.reshape(-1, self.c3d, self.xd, self.D, self.zd)
        x = self.conv_blocks(x)
        x = self.output(x)
        return x

model = M4().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"M4 params: {n_params/1e6:.2f}M")


M4 params: 0.59M


In [3]:
# ---------------------------------------------------------------------------
# Weighted spectral loss
# ---------------------------------------------------------------------------
# L = L_MSE + lambda * L_spectral
#
# L_spectral = sum_k  W[k] * |U_k - V_k|^2  /  sum_k W[k]
#   - U_k, V_k : unitary 2D FFT of prediction / target over the periodic
#                wall-parallel axes (x, z), per (component, y, batch).
#   - W[k]     : FIXED weight, derived from the TARGET spectrum (detached).
#
# Weighting scheme: "equal weight per log-k band".
#   Band b  = { k : k_b <= |k| < k_{b+1} }, edges log-spaced.
#   w_b     = 1 / (N_b * E_b)
#     N_b   = # of wavenumbers in band b   (corrects the 2D "ring area" growth)
#     E_b   = mean target energy in band b = (1/N_b) * sum_{k in b} |V_k|^2
#   => every band contributes ~equally to the loss, so the network spends
#      real gradient budget on the small scales that plain MSE swamps.
# ---------------------------------------------------------------------------

class SpectralLoss(nn.Module):
    def __init__(self, target, dim=(-2, -1), n_bands=6, kmin=1.0, lambda_weight=1.0):
        """
        target       : a batch of targets in the SAME layout as the model output,
                       shape [B, C, X, Y, Z]. Used (detached) to build W[k].
        dim          : the two periodic axes to FFT over (x, z) = (-2, -1).
        n_bands      : number of log-spaced |k| bands (k=0/mean is dropped).
        kmin         : smallest |k| included in banding.
        lambda_weight: coefficient of L_spectral in the total loss.
        """
        super().__init__()
        assert lambda_weight >= 0
        self.dim = dim
        self.lambda_weight = lambda_weight
        self.W = self._build_weights(target, dim, n_bands, kmin)   # [1,1,X,Z], detached

    # -- build the fixed per-mode weight map from the target spectrum --------
    def _build_weights(self, target, dim, n_bands, kmin):
        with torch.no_grad():
            target = target.detach()
            spec = torch.fft.fft2(target, dim=dim, norm="ortho")
            energy = (spec.abs() ** 2)                 # [B,C,X,Y,Z]
            # Average the energy over batch, component and y -> [X, Z]
            E = energy.mean(dim=(0, 1, 3))             # [X, Z]

            # |k| of each mode. fftfreq gives cycles/sample; scale to "radians/sample"
            # by 2*pi so that k=1 is the first non-DC mode (matches kmin=1.0).
            Xn, Zn = E.shape
            kx = torch.fft.fftfreq(Xn, device=E.device) * (2 * np.pi)
            kz = torch.fft.fftfreq(Zn, device=E.device) * (2 * np.pi)
            KX, KZ = torch.meshgrid(kx, kz, indexing="ij")
            k_mag = torch.hypot(KX, KZ)                # [X, Z]

            # log-spaced band edges between kmin and max |k|
            k_max = k_mag.max()
            edges = torch.logspace(
                torch.log10(torch.tensor(kmin)),
                torch.log10(k_max),
                n_bands + 1,
                device=E.device,
            )

            W = torch.zeros_like(k_mag)
            band_E, band_N = [], []
            for b in range(n_bands):
                lo, hi = float(edges[b]), float(edges[b + 1])
                mask = (k_mag >= lo) & (k_mag < hi)
                N_b = max(int(mask.sum()), 1)
                E_b = E[mask].mean().clamp(min=1e-12)
                W[mask] = 1.0 / (N_b * E_b)
                band_E.append(float(E_b))
                band_N.append(N_b)
            # drop the DC / mean mode (k=0) - it carries the profile, not the cascade
            W[k_mag == 0] = 0.0
            self.band_info = list(zip([f"{lo:.2f}-{hi:.2f}" for lo, hi in zip(edges[:-1], edges[1:])], band_N, band_E))
            # [1,1,X,1,Z]: the singleton over Y so it broadcasts against [B,C,X,Y,Z]
            return W[None, None, :, None, :]

    def forward(self, pred, target):
        spec_pred = torch.fft.fft2(pred, dim=self.dim, norm="ortho")
        spec_tgt = torch.fft.fft2(target, dim=self.dim, norm="ortho")
        diff2 = (spec_pred - spec_tgt).abs() ** 2       # [B,C,X,Y,Z]
        # W is [1,1,X,Z]; broadcast over B, C, Y
        weighted = (diff2 * self.W).sum() / self.W.sum()
        mse = torch.nn.functional.mse_loss(pred, target)
        return mse + self.lambda_weight * weighted, mse, weighted


# Build it from a validation batch so the weights come from real target statistics.
xb, yb = next(iter(val_dl))
spectral = SpectralLoss(yb, dim=(-2, -1), n_bands=6, kmin=1.0, lambda_weight=1.0).to(device)

print("W shape:", tuple(spectral.W.shape), " device:", spectral.W.device)
print(f"{'band |k|':>14}  {'#modes':>7}  {'mean target E':>14}  {'w_b':>10}")
for name, N_b, E_b in spectral.band_info:
    print(f"{name:>14}  {N_b:>7d}  {E_b:>14.3e}  {1.0/(N_b*E_b):>10.3e}")


W shape: (1, 1, 24, 1, 24)  device: cuda:0
      band |k|   #modes   mean target E         w_b
     1.00-1.28       24       2.238e+00   1.861e-02
     1.28-1.64       52       1.702e+00   1.130e-02
     1.64-2.11       76       1.039e+00   1.267e-02
     2.11-2.70      144       1.010e+00   6.872e-03
     2.70-3.47      170       5.961e-01   9.868e-03
     3.47-4.44       64       1.445e-03   1.081e+01


In [4]:
# ---------------------------------------------------------------------------
# Verification
#  (a) Parseval: unweighted spectral L2 == physical L2  (sanity check of the FFT path)
#  (b) The weighted loss actually changes per-band emphasis (it is NOT a no-op)
#  (c) gradient flows through the FFT into the prediction
# ---------------------------------------------------------------------------

xb, yb = next(iter(val_dl))            # real normalized target batch
pred = model(xb)                       # model output, same layout as yb

# (a) Parseval on a random pair
rng = torch.randn_like(yb)
phys_l2  = ((rng - yb) ** 2).mean()
spec = torch.fft.fft2(rng - yb, dim=(-2, -1), norm="ortho")
spec_l2  = (spec.abs() ** 2).mean()
print("(a) Parseval  physical L2 =", float(phys_l2), " spectral L2 =", float(spec_l2),
      " rel diff =", float((phys_l2 - spec_l2).abs() / phys_l2))

# (b) per-band contribution of the ERROR under W=1 (plain MSE) vs W=equal-log-band
err = (pred - yb)
err_spec = torch.fft.fft2(err, dim=(-2, -1), norm="ortho")
err_E = (err_spec.abs() ** 2).mean(dim=(0, 1, 3))      # [X,Z] error energy per mode

kx = torch.fft.fftfreq(err_E.shape[0], device=err_E.device) * (2 * np.pi)
kz = torch.fft.fftfreq(err_E.shape[1], device=err_E.device) * (2 * np.pi)
KX, KZ = torch.meshgrid(kx, kz, indexing="ij")
k_mag = torch.hypot(KX, KZ)

# reuse the same band edges as the loss
edges = torch.logspace(torch.log10(torch.tensor(1.0)), torch.log10(k_mag.max()), 7, device=err_E.device)
print("\n(b) per-band ERROR energy  (where the model's error actually lives)")
print(f"{'band |k|':>14}  {'plain-MSE share %':>18}  {'weighted share %':>18}")
plain_tot, weighted_tot = err_E.sum(), 0.0
# extract the [X, Z] weight map (W is [1,1,X,1,Z]) for the per-band share table
W = spectral.W[0, 0, :, 0, :]
plain_shares, weighted_shares = [], []
for b in range(6):
    lo, hi = float(edges[b]), float(edges[b + 1])
    mask = (k_mag >= lo) & (k_mag < hi)
    plain_shares.append(float(err_E[mask].sum() / err_E.sum() * 100))
    weighted_shares.append(float((err_E[mask] * W[mask]).sum()))
weighted_tot = sum(weighted_shares)
for b, name in enumerate([f"{float(edges[b]):.2f}-{float(edges[b+1]):.2f}" for b in range(6)]):
    print(f"{name:>14}  {plain_shares[b]:>17.2f}%  {weighted_shares[b]/weighted_tot*100:>17.2f}%")

# (c) gradient through the spectral path
p = pred.detach().requires_grad_(True)
t_loss, _, _ = spectral(p, yb)
t_loss.backward()
print("\n(c) grad w.r.t. pred exists:", p.grad is not None,
      "  ||grad|| =", float(p.grad.norm()))


(a) Parseval  physical L2 = 2.1508419513702393  spectral L2 = 2.1508419513702393  rel diff = 0.0

(b) per-band ERROR energy  (where the model's error actually lives)
      band |k|   plain-MSE share %    weighted share %
     1.00-1.28               8.11%              16.86%
     1.28-1.64              13.28%              16.76%
     1.64-2.11              11.70%              16.56%
     2.11-2.70              21.57%              16.56%
     2.70-3.47              14.99%              16.53%
     3.47-4.44               0.01%              16.72%

(c) grad w.r.t. pred exists: True   ||grad|| = 0.1888139247894287


In [5]:
# ---------------------------------------------------------------------------
# Training config
# ---------------------------------------------------------------------------
epochs = 20
lr = 5e-4
optim = torch.optim.Adam
batch_size = 4
project_name = "framework-1"

# spectral-loss weight in the TRAINING objective.
# Keep small: the per-band weights already amplify the fine scales ~500x,
# so lambda should be modest to avoid wrecking overall accuracy.
spectral_lambda = 0.2

# training loss = MSE + lambda * spectral
train_loss = SpectralLoss(yb, dim=(-2, -1), n_bands=6, kmin=1.0,
                          lambda_weight=spectral_lambda).to(device)

# fixed band edges for the full-val diagnostic (recomputed from the grid)
XN, ZN = 24, 24
_kx = torch.fft.fftfreq(XN) * (2 * np.pi)
_kz = torch.fft.fftfreq(ZN) * (2 * np.pi)
_KX, _KZ = torch.meshgrid(_kx, _kz, indexing="ij")
_KMAG = torch.hypot(_KX, _KZ)
BAND_EDGES = torch.logspace(
    torch.log10(torch.tensor(1.0)),
    torch.log10(_KMAG.max()),
    7,  # n_bands + 1 edges
).tolist()
print("bands:", [f"{BAND_EDGES[i]:.2f}-{BAND_EDGES[i+1]:.2f}" for i in range(6)])
print(f"train loss = MSE + {spectral_lambda} * spectral")


bands: ['1.00-1.28', '1.28-1.64', '1.64-2.11', '2.11-2.70', '2.70-3.47', '3.47-4.44']
train loss = MSE + 0.8 * spectral


In [6]:
# ---------------------------------------------------------------------------
# Train / eval / full-val spectral diagnostic
# ---------------------------------------------------------------------------

def train(model, opti, loss_fn, ds):
    """One epoch. loss_fn(pred, target) -> (total, mse, spectral)."""
    model.train()
    tot = mse_tot = spec_tot = 0.0
    for x, y in ds:
        opti.zero_grad()
        total, mse, spec = loss_fn(model(x), y)
        total.backward()
        opti.step()
        tot += total.item(); mse_tot += mse.item(); spec_tot += spec.item()
    n = len(ds)
    return tot / n, mse_tot / n, spec_tot / n


@torch.no_grad()
def evaluate(model, ds):
    """Normalized MSE on val (== 1 - EV, since the fluc space has unit variance)."""
    model.eval()
    tot = 0.0
    for x, y in ds:
        tot += torch.nn.functional.mse_loss(model(x), y).item()
    return tot / len(ds)


@torch.no_grad()
def val_spectral_bands(model, ds, edges=BAND_EDGES, dim=(-2, -1)):
    """Per-band error energy over the FULL validation set (diagnostic, not a loss).

    Returns a dict {band_name: error_energy_share_%} where the share is the
    fraction of total val error energy falling in that |k| band. Low share in
    the fine bands => the model is resolving them; high share => it is failing
    exactly where plain MSE gives up.
    """
    model.eval()
    Xn, Zn = ds.dataset.y.shape[2], ds.dataset.y.shape[4]
    kx = torch.fft.fftfreq(Xn, device=device) * (2 * np.pi)
    kz = torch.fft.fftfreq(Zn, device=device) * (2 * np.pi)
    KX, KZ = torch.meshgrid(kx, kz, indexing="ij")
    k_mag = torch.hypot(KX, KZ)
    k_mag = k_mag.to(device)

    band_err = [0.0] * (len(edges) - 1)
    for x, y in ds:
        err = model(x) - y
        spec = torch.fft.fft2(err, dim=dim, norm="ortho")
        E = (spec.abs() ** 2).mean(dim=(0, 1, 3))          # [X, Z] error energy
        for b in range(len(edges) - 1):
            mask = (k_mag >= edges[b]) & (k_mag < edges[b + 1])
            band_err[b] += float(E[mask].sum())
    tot = sum(band_err)
    return {f"{edges[b]:.2f}-{edges[b+1]:.2f}": 100.0 * band_err[b] / tot
            for b in range(len(edges) - 1)}


@torch.no_grad()
def val_band_skill(model, ds, edges=BAND_EDGES, dim=(-2, -1)):
    """Per-band skill = 1 - E_err(b)/E_target(b)  (== EV restricted to band b).

    Denominated by the band's OWN target energy (not the total error), so bands
    are directly comparable and the trivial-zero illusion is removed:
      skill ~ 1  : either genuinely good, OR predicting ~0 on a small signal (check E_err)
      skill < 0  : worse than the mean at that scale (anti-correlated) - real failure
    """
    model.eval()
    Xn, Zn = ds.dataset.y.shape[2], ds.dataset.y.shape[4]
    kx = torch.fft.fftfreq(Xn, device=device) * (2 * np.pi)
    kz = torch.fft.fftfreq(Zn, device=device) * (2 * np.pi)
    KX, KZ = torch.meshgrid(kx, kz, indexing="ij")
    k_mag = torch.hypot(KX, KZ).to(device)

    band_err = [0.0] * (len(edges) - 1)
    band_tgt = [0.0] * (len(edges) - 1)
    for x, y in ds:
        pred = model(x)
        E_err = (torch.fft.fft2(pred - y, dim=dim, norm="ortho").abs() ** 2).mean(dim=(0, 1, 3))
        E_tgt = (torch.fft.fft2(y, dim=dim, norm="ortho").abs() ** 2).mean(dim=(0, 1, 3))
        for b in range(len(edges) - 1):
            mask = (k_mag >= edges[b]) & (k_mag < edges[b + 1])
            band_err[b] += float(E_err[mask].sum())
            band_tgt[b] += float(E_tgt[mask].sum())
    return {f"{edges[b]:.2f}-{edges[b+1]:.2f}": 1.0 - band_err[b] / band_tgt[b]
            for b in range(len(edges) - 1)}


def train_loop(model, loss_fn, ds_train, ds_val, epochs, optim, lr, project_name):
    opti = optim(model.parameters(), lr=lr)
    trackio.init(
        project=project_name,
        config={"epochs": epochs, "learning_rate": lr,
                "spectral_lambda": loss_fn.lambda_weight},
    )
    # full-val spectral diagnostic at init (untrained model) - baseline
    with torch.no_grad():
        base_bands = val_spectral_bands(model, ds_val)
        base_skill = val_band_skill(model, ds_val)
    trackio.log({"epoch": -1,
                 **{f"val/err_share_%/{n}": v for n, v in base_bands.items()},
                 **{f"val/skill/{n}": v for n, v in base_skill.items()}})

    for epoch in range(epochs):
        train_err, train_mse, train_spec = train(model, opti, loss_fn, ds_train)
        val_mse = evaluate(model, ds_val)                       # == 1 - EV
        val_ev = 1.0 - val_mse                                   # unit-var fluc space
        bands = val_spectral_bands(model, ds_val)               # where the error lives
        skill = val_band_skill(model, ds_val)                   # how good per band is

        trackio.log({
            "epoch": epoch,
            "train/loss": train_err,
            "train/mse": train_mse,
            "train/spectral": train_spec,
            "val/mse": val_mse,
            "val/EV": val_ev,
            **{f"val/err_share_%/{name}": v for name, v in bands.items()},
            **{f"val/skill/{name}": v for name, v in skill.items()},
        })
    trackio.finish()


In [7]:
# ---------------------------------------------------------------------------
# Run  (smoke test: 2 epochs; bump `n_epochs` once the curve looks sane)
# ---------------------------------------------------------------------------
n_epochs = 20
train_loop(
    model=model,
    loss_fn=train_loss,
    ds_train=train_dl,
    ds_val=val_dl,
    epochs=n_epochs,
    optim=optim,
    lr=lr,
    project_name=project_name,
)


* Trackio project initialized: framework-1
* Trackio metrics logged to: /home/chancrin/.cache/huggingface/trackio
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: jolly-valley-7


* Run finished. Uploading logs to Trackio (please wait...)
